In [1]:
import pandas as pd

# Load your existing dataset (change the filename)
df = pd.read_excel("data/raw/total_count_intentional_homicide.xlsx", header=2)

# Standardise column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Standardise alternative column names
df = df.rename(columns={
    "unit_of_measurements": "unit",
    "unit_of_measurement": "unit",
    "values": "value"
})

print(df.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/total_count_intentional_homicide.xlsx'

In [3]:
# Standardise text values
for col in ["indicator", "dimension", "category",
            "sex", "age", "unit"]:
    df[col] = df[col].astype("string").str.strip()

df["value"] = pd.to_numeric(df["value"], errors="coerce")
df["year"] = pd.to_numeric(df["year"], errors="coerce")

# Only intentional homicide victims, 2015–2024
data = df[
    (df["indicator"] == "Victims of intentional homicide") &
    (df["year"].between(2015, 2024)) &
    (df["value"].notna())
].copy()

# Separate counts and rates
counts = data[
    data["unit"].str.lower() == "counts"
].copy()

rates = data[
    data["unit"].str.lower().str.startswith("rate")
].copy()

In [4]:
# Use total sex and age to avoid counting the same
# category multiple times across demographic breakdowns
breakdowns = counts[
    (counts["dimension"] != "Total") &
    (counts["category"].notna()) &
    (counts["category"] != "Total") &
    (counts["sex"] == "Total") &
    (counts["age"] == "Total")
]

# Count distinct available categories within each dimension
dimension_coverage = (
    breakdowns
    .groupby(["country", "year", "dimension"])["category"]
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
)

print(dimension_coverage.head())

dimension  country  year  by mechanisms  by relationship to perpetrator  \
0          Albania  2015              3                               6   
1          Albania  2016              3                               6   
2          Albania  2017              2                               6   
3          Albania  2018              2                               6   
4          Albania  2019              3                               1   

dimension  by situational context  
0                               3  
1                               3  
2                               3  
3                               3  
4                               1  


In [5]:
# Sex coverage
sex_coverage = (
    counts[
        (counts["dimension"] == "Total") &
        (counts["category"] == "Total") &
        (counts["age"] == "Total") &
        (counts["sex"] != "Total")
    ]
    .groupby(["country", "year"])["sex"]
    .nunique()
    .rename("sex_categories")
    .reset_index()
)

# Age coverage
age_coverage = (
    counts[
        (counts["dimension"] == "Total") &
        (counts["category"] == "Total") &
        (counts["sex"] == "Total") &
        (counts["age"] != "Total")
    ]
    .groupby(["country", "year"])["age"]
    .nunique()
    .rename("age_categories")
    .reset_index()
)

In [6]:
overall_rates = rates[
    (rates["dimension"] == "Total") &
    (rates["category"] == "Total") &
    (rates["sex"] == "Total") &
    (rates["age"] == "Total")
][["country", "year", "value"]].copy()

overall_rates = overall_rates.rename(
    columns={"value": "overall_rate"}
)

In [7]:
print(
    overall_rates[
        overall_rates.duplicated(["country", "year"], keep=False)
    ]
)

Empty DataFrame
Columns: [country, year, overall_rate]
Index: []


In [8]:
availability = (
    overall_rates
    .merge(dimension_coverage, on=["country", "year"], how="left")
    .merge(sex_coverage, on=["country", "year"], how="left")
    .merge(age_coverage, on=["country", "year"], how="left")
)

# Replace missing coverage counts with zero
coverage_cols = [
    col for col in availability.columns
    if col not in ["country", "year", "overall_rate"]
]

availability[coverage_cols] = (
    availability[coverage_cols]
    .fillna(0)
    .astype(int)
)

availability = availability.sort_values(["year", "country"])

print(availability.head(20))

                             country  year  overall_rate  by mechanisms  \
0                        Afghanistan  2015      9.952186              0   
2                            Albania  2015      2.207938              3   
38                           Algeria  2015      1.364334              4   
7                     American Samoa  2015      7.564583              0   
3                            Andorra  2015      0.000000              3   
1                             Angola  2015      4.453473              0   
5                          Argentina  2015      6.525287              3   
6                            Armenia  2015      2.567458              6   
8                          Australia  2015      0.985468              5   
9                            Austria  2015      0.509021              3   
10                        Azerbaijan  2015      2.163470              0   
15                           Bahamas  2015     37.887979              4   
14                       

In [ ]:
availability.to_csv(
    "outputs/table/homicide_availability.csv",
    index=False
)

In [10]:
yearly_coverage = (
    availability
    .groupby("year")[coverage_cols]
    .agg(lambda x: (x >= 2).sum())
)

print(yearly_coverage)

      by mechanisms  by relationship to perpetrator  by situational context  \
year                                                                          
2015             64                              67                      37   
2016             74                              65                      37   
2017             79                              74                      40   
2018             74                              75                      44   
2019             73                              75                      48   
2020             72                              72                      45   
2021             75                              73                      50   
2022             69                              71                      48   
2023             65                              70                      48   
2024             57                              62                      42   

      sex_categories  age_categories  
year        